# Interface and capability map

```{figure} ../figures/mermaid-351c07eaa8d0.svg
:name: mermaid-351c07eaa8d0
:alt: Methods and attributes of a fitted model, including path truncation, learner selection, coefficient aggregation, prediction types, and diagnostics.

The fitted-model interface exposes stopping, risk, selected learners, coefficients, predictions, and diagnostics from one stored boosting path.
```

This page is the first of the Advanced group. It is for readers who have run the
[Quickstart](quickstart.ipynb) and need to know, before choosing a fitting
function, family or base learner, what each interface accepts and which
operations a fitted model supports. It covers the four fitting functions, the
formula notation, the learners and families each function accepts, the methods
of a fitted model, and how to query the capability table from code. The next
page, [Choosing terms from the data](choosing_terms.ipynb), covers automatic
construction of a formula from the observed columns.

The family determines the loss and available distributional operations. The
fitting function determines which base learners may enter the model. Some
diagnostics also require distribution functions available only for certain
families.



## Choosing a fitting function

Four fitting functions are available, and the choice follows from the effects and response parameters the model needs. What they do statistically is
derived in [](../methodology/overview.ipynb); what follows is the operational
difference.

| Function | Use it when | Each candidate effect is |
|---|---|---|
| `glmboost` | You want linear effects only, and expect a sparse subset of many predictors. | One column: a single coefficient per predictor. |
| `gamboost` | You need structured effects — smooth curves, surfaces, group effects, trees. | A block of columns with its own penalty. |
| `glmboostLSS` | You want linear effects on the response mean, spread, or shape. | One column, separately for each distribution parameter. |
| `gamboostLSS` | Same, with structured effects. | A block of columns, separately for each distribution parameter. |



## The formula interface

Models are specified with a formula, the model-description notation from R,
accepted in both languages:

```
y ~ bbs(x1) + bols(x2) + brandom(group)
```

The response goes on the left of `~`. Each term on the right names one **base
learner**: a small, self-contained model for one variable or one group of
variables. `bbs(x1)` is a smooth curve in `x1`, `bols(x2)` is a straight-line
effect in `x2`, and `brandom(group)` is a per-group offset.

Each `+`-separated term is a *candidate*. Every iteration, the algorithm fits all
of the candidates to the current residual, keeps the one that fits best, and adds
a fraction of it to the model. A candidate never chosen contributes nothing,
which is how a formula with hundreds of terms still produces a small model. This
Unlike `lm` and `glm`, where `+` terms are estimated jointly, the formula
terms compete for selection across boosting iterations.

A term is one candidate however many columns it spans. `bbs(x1)` may use twenty
spline basis columns and `brandom(group)` one column per group, but each is
selected or rejected as a unit, with a single penalty and a single effective
degrees-of-freedom setting. `glmboost` instead selects individual linear
components, one predictor column at a time.



## Families and learners accepted by each fitting function

- `glmboost` fits one predictor for a standard regression, classification, count, ordinal, or survival family. Each candidate is a `Bols` linear component. Smooth, spatial, grouped, tree, and user-defined effects are unavailable.

- `gamboost` fits one additive predictor for a standard family. It accepts `Bols`, `Bbs`, `Bmono`, `Btree`, `Bmrf`, `Bspatial`, `Brandom`, `Bns`, `Bkernel`, `Bconstrained`, and `Buser`; the definitions and matrix requirements appear below.

- `glmboostLSS` fits one linear predictor for every parameter of a distributional family. Each candidate is a `Bols` linear component, and every parameter has its own selection path.

- `gamboostLSS` fits one additive predictor for every distribution parameter. It accepts `Bols`, a linear effect optionally including an intercept and ridge penalty; `Bbs`, a difference-penalized B-spline; `Bns`, a B-spline basis with an identity ridge penalty; `Brandom`, a ridge-penalized random intercept with one indicator column per group; `Bmrf`, a graph-smoothed categorical effect using a supplied neighborhood penalty; `Bspatial`, a tensor-product P-spline surface for two predictors; `Bmono`, a P-spline constrained to be increasing, decreasing, convex, concave, positive, or negative; `Bconstrained`, a P-spline subject to a caller-supplied linear constraint matrix; and `Btree`, a decision stump or shallow regression tree. It also accepts `Bkernel`, a kernel-ridge effect built from a supplied kernel matrix, and `Buser`, an effect built from supplied design and penalty matrices. All eleven may compete in one parameter formula. `glmboostLSS` is the restricted case: it fits one linear term per predictor, so it takes `Bols` or a bare column name and rejects every learner that expands a predictor into a basis.

The `gamboost` base learners are:

- `Bols`: a linear effect, optionally with an intercept and ridge penalty;
- `Bbs`: a penalized B-spline for a smooth univariate effect;
- `Bmono`: a shape-constrained P-spline for monotonicity, convexity, concavity, positivity, or negativity;
- `Btree`: a decision stump by default, or a shallow regression tree when `maxdepth` is increased;
- `Bmrf`: a Markov-random-field effect for a categorical variable with a supplied neighborhood penalty;
- `Bspatial`: a tensor-product P-spline surface for two predictors;
- `Brandom`: a ridge-penalized random effect represented by one indicator column per group;
- `Bns`: a B-spline basis without the difference penalty used by `Bbs`;
- `Bkernel`: a kernel-ridge effect using an explicit dense kernel matrix;
- `Bconstrained`: a P-spline subject to a caller-supplied linear constraint matrix; and
- `Buser`: an arbitrary caller-supplied design matrix and penalty matrix.

Within each `gamboostLSS` parameter predictor, every formula term is a separate candidate learner. For `mu = y ~ bbs(x) + bols(z)`, one boosting update fits both candidates to the current negative gradient for `mu`, compares their weighted residual sums of squares, and adds only the better update. A term such as `brandom(group)` enters the same competition as one learner: its coefficient block contains one ridge-penalized effect per fitted group, so selection chooses or rejects the grouped effect as a unit. The scale or shape predictors have their own formulas, candidates, coefficients, and stopping counts. Cyclic fitting performs one such selection for every active parameter per cycle; noncyclic fitting compares the best update from every parameter and spends one global step on the largest joint-risk reduction.

Cross-validation and stability selection represent resamples through observation
weights. Learners reuse their spline bases, group layouts or tree settings,
but recompute weighted centering and the smoothing penalty implied by `df` for
each resample, following R. Prediction reuses the fitted basis definition,
including its training knots, when evaluating the training data, new rows or
a shorter boosting path.

`Bconstrained` uses a caller-supplied constraint matrix, which has no upstream
counterpart; it is checked for feasibility, replay, and resampling. Upstream
`mboost::bns()` reports that the learner was removed and recommends `bbs`,
whereas mboostrs implements a ridge-penalized B-spline with that name.
Everything else — mixed `Bols`/`Bbs`, `Brandom`, `Bmrf`, `Bspatial`, `Bmono`,
and `Btree` — is checked numerically against R `gamboostLSS`. The internals
summarized here are documented in
[](../implementation/base_learners.ipynb).



## Family capabilities and limitations



### Standard families

The implemented standard families fit a single additive predictor and support link-scale prediction. Gaussian, count, robust-regression, quantile, expectile, binary, ordinal, survival, and multinomial losses are available, but their post-fit operations differ.

- `Gaussian`, `GaussReg`, `Poisson`, `Laplace`, `GammaReg`, `Quantile`, and `Expectile` support fitting plus link- and response-scale prediction. `Huber`, `NBinomial`, and `PropOdds` also fit and predict, but their residual and diagnostic surfaces are less complete.
- `Binomial`, `AdaExp`, `HingeLoss`, `PropOdds`, and `Multinomial` do not currently advertise a direct class-prediction operation in the capability contract. Obtain response probabilities or margins and apply an explicit, documented decision rule. For binomial data, Python accepts 0/1 responses even though the Rust loss uses -1/+1 coding internally.
- `CoxPH`, `Weibull`, `Loglog`, and `Lognormal` accept survival responses. AIC is intentionally unavailable for these families. A `survfit()` method is available in Python on `GLMBoostModel` and `GAMBoostModel`; the R frontend does not export one. Cox response-scale prediction is only partial; link-scale risk scores are the stable common output.
- `Multinomial` uses the Kronecker model path. Coefficients and link/response predictions are available, but residuals, AIC, and hat values are not.

The standard-family AIC, log-likelihood, and diagnostics columns are marked partial where they rely on the boosting path's effective-degrees-of-freedom approximation or where the family lacks a complete distributional helper surface. Cross-validation is the general stopping rule; AIC is not a universal substitute.

`AUC`, `Cindex`, `Gehan` and `Hurdle` are implemented families. Their
resampling and validation coverage differs: `Cindex` accepts zero-one
case weights and rejects bootstrap multiplicities; weighted `AUC` and
`Gehan` resampling is smoke-tested rather than verified against R. See
[](families.ipynb) and [](survival.ipynb) for their prediction scales.
The upstream helpers `RCG`, `FP`, `IPCweights` and the generic `Family`
constructor remain unsupported.



(interface-distributional-families)=

### Distributional families

An LSS fit estimates several linked parameters, so fitting support and diagnostic support must be considered separately.

- `GaussianLSS`, `NBinomialLSS`, `ZIPoLSS`, `PoissonLSS`, `GeometricLSS`, `BinomialLSS`, `NegativeBinomialLSS`, `BCTLSS`, and `BCPELSS` provide density or mass, CDF, quantile, simulation, and deviance-increment helpers. They support the complete distributional diagnostic chain: PIT or quantile residuals, prediction intervals, centiles, and coverage summaries.
- `GammaLSS`, `StudentTLSS`, `BetaLSS`, and `ZINBLSS` also provide CDF and quantile helpers, so the same diagnostic chain works for them. These four are computed from the regularized incomplete gamma and beta functions implemented in `gamlss.py`, because SciPy is not a runtime dependency of this package; they are validated against SciPy as a test-time oracle in `tests/test_lss_special_functions.py`. The Student-t quantile returns the location exactly at `p = 0.5`: its CDF is expressed through `betainc(df/2, 1/2, df/(df + t^2))`, whose last argument rounds to 1 for `|t|` below `sqrt(df * eps)`, leaving the CDF numerically flat around the median.
- `GumbelLSS`, `LogisticLSS`, `InverseGaussianLSS`, `Pareto2LSS`, `ZeroAdjustedPoissonLSS`, `ZeroAdjustedGammaLSS`, `BCCGLSS`, `JSULSS`, and the survival LSS families fit and predict, while most distributional helper operations remain planned. These families are available for modelling and lack the full set of diagnostic helpers.
- `PIGLSS` fits and predicts on the response scale, with exact density and gradients. It does not provide quantile helpers, so `centiles` raises `NotImplementedError` for it. `DirichletLSS` supports between 2 and 20 components; a univariate CDF is not meaningful for that response.

`update()` refits the stored specification. It accepts `mstop`, `weights`,
`risk`, and `trace`; the latter two are forwarded to `boost_control`, and omitting
them carries the original fit's settings over. It rejects `oobweights`, because
neither `glmboostLSS` nor `gamboostLSS` takes a separate out-of-bag weight vector
— use `risk="oobag"` with zero weights on the held-out rows instead.

For every LSS model, `risk_` is the joint negative-log-likelihood path. It is not a collection of independently comparable per-parameter risks. Changing `mstop` refits the interleaved parameter updates rather than merely slicing one stored scalar path.



## Base-learner boundaries

New-data prediction is implemented for every `gamboost` learner listed above, but construction and resampling have several qualifications:

- `Bmrf` requires a neighborhood matrix `K`, and `Bconstrained` requires a constraint matrix `C`. In Python, bind the matrix name through `env`, for example `bmrf(region, K=K_regions)` with `env={"K_regions": K}`. The R frontend resolves `K=` or upstream-style `bnd=` from the formula environment. `Bconstrained` still requires its matrix to be supplied programmatically.
- `Buser` requires compatible user-supplied design and penalty matrices. `Bkernel` fits a kernel-ridge effect from a dense kernel matrix you supply. R's `mboost::bkernel` is built for pathway and genome-wide association analysis and constructs its kernel from genotype data; this learner does not reproduce that construction, so a model written against upstream `bkernel` will not transfer unchanged.
- Cross-validation and stability selection preserve the original observation order and express each resample through weights. This makes an explicit `Buser` training matrix safe to reuse while still forcing weighted centering and `df`-to-`lambda` conversion to be recomputed for every resample.
Per-learner requirements when predicting on new data:

| Learner | What you must supply, and what to expect |
|---|---|
| `Buser` | An explicit matrix keyed by learner name, for every prediction and partial effect. The fit owns its training `X` and `K`, but never guesses a basis from data-frame columns and never reuses training rows. |
| `Bkernel` | Nothing extra; prediction builds a cross-kernel between the new and training observations. |
| `Brandom` | An integer group code seen during fitting. An unseen code raises an unknown-level error. |
| `Bmrf` | A zero-based region code indexing a row of `K`. An unknown code raises. |
| `Bspatial` | Both coordinate columns. Training boundary knots are reused. Partial effects return coordinate pairs, since a surface has no single x-axis. |
| `Btree` | Nothing extra. Coefficients come back as per-step encoded node vectors, because different updates may contain different numbers of nodes. |

Constrained solvers:

- `Bmono` and `Bconstrained` solve each step's constrained quadratic programme
  exactly by default (`type="quad.prog"`, upstream `bmono`'s default), so the
  constraint holds at every step. `type="iterative"` uses a violation penalty
  instead: faster, and it leaves a small same-signed violation each step that a
  long path accumulates.
- `Bconstrained` requires a finite, nonempty `C` with `knots + degree + 1`
  columns.

One frontend difference: Python `Btree` accepts several splitting variables in
one term where R accepts one.

- `Btree` is available inside `gamboost`, but it does not implement upstream `blackboost`: it splits on squared error, whereas upstream's `btree` uses a conditional-inference criterion. `mboost::bss()` was withdrawn and now raises an error referring users to `bbs()`, as does `mboost::bns()`. `brad` is implemented as a Matérn(3/2) radial basis that differs from upstream only in how it chooses its centres.



## Fitted model methods

A fitted model stores each iteration's chosen learner and coefficient update.
For `glmboost` and `gamboost`, summing the first `k` updates gives the
coefficients and predictions at `mstop=k` without refitting. The stored
selection sequence also shows when each term entered the model.

The things you can ask a fitted model:

| You want | Call | Notes |
|---|---|---|
| the stopping iteration and risk path | `mstop`, `risk_` | Risk is in-bag by default. |
| which learners were chosen, and when | `selected()` | One entry per iteration. |
| coefficients | `coef()` | Aggregated across the path. |
| fitted values and predictions | `fitted()`, `predict()` | `type="link"` or `"response"`. |
| a shorter path | `model[:k]`, `with_mstop(k)` | No refit for `glmboost`/`gamboost`. |
| a longer path | `continue_to(m)` | Returns the model at a larger `mstop`, equal to a cold fit. Resumes the retained gradient where the fit kept one, and otherwise refits from zero. |
| per-term effect shapes | `partial_effects()` | `gamboost` and `gamboostLSS` only; a `glmboost` term has no shape beyond its coefficient. |
| residual and calibration tables | `residual_frame()`, `calibration_frame()` | |
| prediction intervals | `conformal_interval()` | Standard families. |
| a portable saved model | `save()` | Schema-versioned JSON. |

Truncation differs between model types. `glmboost` and `gamboost` store one
scalar path, so `model[:k]` reads a shorter prefix of it. An LSS model interleaves
updates across distribution parameters, and changing `mstop` changes which
updates happen at all — so `update()` refits rather than slicing.



### Calls that work on every fitted model

`stopping_budget()`, `selected_terms()` and `predict_parameters()` use the same
return shapes for `glmboost`, `gamboost`, `glmboostLSS` and `gamboostLSS` fits,
cyclic or non-cyclic, so callers can use them without branching on model class. A standard model has one predictor, named `f` (mboost's name for the
boosted function); a distributional model has one per parameter.

- `fit.stopping_budget()` returns a `StoppingBudget`: `scheme` (`"single"`,
  `"cyclic"` or `"noncyclic"`), `per_parameter` (updates per predictor) and
  `total` (their sum, the budget a non-cyclic refit takes). `fit.mstop`
  keeps its older shape: an integer, or a mapping when LSS parameters
  differ.
- `fit.selected_terms()` returns one row per update: `order` across the whole
  fit, `parameter`, `iteration` within that parameter, and `learner`. A
  cyclic LSS fit updates every parameter once per iteration in parameter
  order; a non-cyclic fit records the parameter it chose at each update.
- `fit.predict_parameters(newdata, type="link")` returns `{predictor: array}`:
  `{"f": ...}` for a standard model, `{"mu": ..., "sigma": ...}` for a
  Gaussian LSS model. `fit.predict` keeps its older shape: an array for a
  standard model, a mapping for an LSS model unless `parameter=` is given.

```python
for fit in (gam_fit, lss_fit):
    budget = fit.stopping_budget()
    print(budget.scheme, budget.total, budget.per_parameter)
    print(fit.selected_terms(output="records")[:2])
    predictions = fit.predict_parameters(new_rows, type="response")
```

### Methods that raise an error for some families

Some methods exist on the class but reject a particular family, because the
operation needs a quantity that family does not define. For example, a PIT
requires a CDF, and a prediction interval requires a quantile function.

- `aic()` and `hatvalues()` need a usable hat matrix or degrees-of-freedom
  approximation. Unavailable for survival and multinomial fits; partial for some
  structured learners.
- `pit()`, `quantile_residuals()`, and `residual_frame()` on an LSS model need
  the family's CDF. `prediction_interval()` and `centiles()` need its quantile
  function. `deviance_increment()` needs its density or mass function. Families
  that lack these are listed under [](#interface-distributional-families).
- `q_stats()` gives descriptive binned moments. The inferential p-values from the
  full GAMLSS diagnostic are not implemented.
- `worm_plot()` exists on `LSSModel`. The standard model classes provide
  `worm_data()`, the underlying summaries, and no plot method.
- `CvRiskResult` and `StabSelResult` both provide `to_frame()` in Python,
  alongside plotting and array-like access.

The capability table reports implementation availability, R-parity status,
and support for each family separately, with the corresponding test evidence.


In [1]:
import polars as pl
import mboostrs as mb

(
    mb.family_capabilities(output="polars")
    .filter(
        pl.col("name").is_in(
            ["Gaussian", "Binomial", "CoxPH", "Multinomial", "GaussianLSS", "PIGLSS"]
        )
    )
    .select(
        "name",
        "kind",
        "fit",
        "predict_response",
        "aic",
        "cdf",
        "quantile",
        "diagnostics",
        "notes",
    )
)


name,kind,fit,predict_response,aic,cdf,quantile,diagnostics,notes
str,str,str,str,str,str,str,str,str
"""Binomial""","""standard""","""yes""","""yes""","""partial""","""no""","""no""","""partial""","""Rust uses -1/+1 internally; Py…"
"""CoxPH""","""standard_survival""","""yes""","""partial""","""no""","""no""","""no""","""partial""","""AIC is intentionally unavailab…"
"""Gaussian""","""standard""","""yes""","""yes""","""partial""","""no""","""no""","""partial""",""" AIC uses the per-observation …"
"""GaussianLSS""","""lss""","""yes""","""yes""","""no""","""yes""","""yes""","""partial""",""""""
"""Multinomial""","""standard_multiclass""","""yes""","""yes""","""no""","""no""","""no""","""partial""",""""""
"""PIGLSS""","""lss""","""yes""","""yes""","""no""","""planned""","""planned""","""planned""","""exact density and gradients vi…"


## Querying capabilities from code

Query the capability table directly when code needs to branch over families
or learners, as shown in the cell above. It returns
`yes`, `partial`, `planned`, or `no` for whether an operation is functionally
available.

Separate columns record how well each operation is *evidenced* per frontend:

- `verified`: numerical parity is checked against the stated oracle;
- `smoke`: the operation runs, without a full numerical parity claim;
- `intentional-deviation`: behavior differs from upstream on purpose, and the
  reason is recorded in the row;
- `unverified`: an implementation exists but the evidence does not;
- `unsupported`: the operation is not available and must not be advertised.

`output="records"` returns plain dictionaries for serialization or environments
without a dataframe library. `output="auto"` uses Polars when installed.
[](../appendix/capabilities.ipynb) gives the full tables, including validation
IDs and tolerances.

## See also

- [](families.ipynb) explains how each loss, link, and response type changes the model.
- [](base_learners.ipynb) derives the design and penalty for every supported learner.
- [](model_selection.ipynb) covers cross-validation, stability selection, and the valid scope of AIC.
- [](diagnostics.ipynb) shows which distributional helpers each diagnostic consumes.
- [](../appendix/capabilities.ipynb) gives the full capability and validation tables.
